<a href="https://colab.research.google.com/github/carolinampessoa/TechChallengeFase5/blob/main/TechChallengeFase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#Instalação de libs

!pip install openai
!pip install reportlab

In [7]:
#Configurar API Key (uso de LLM da OpenAI)

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

Digite sua OpenAI API Key: ··········


In [8]:
#Importar imagem para avaliação

from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


Saving Screenshot_1.png to Screenshot_1.png
Imagem carregada: Screenshot_1.png


In [9]:
# Funções auxiliares

from openai import OpenAI
import base64
import json

client = OpenAI()

def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

base64_image = encode_image(image_path)

In [10]:
# Extração Livre de Componentes

free_extraction_prompt = """
Analise o diagrama de arquitetura de software presente na imagem.

Identifique TODOS os elementos arquiteturais semanticamente relevantes.

Não restrinja a categorias pré-definidas.

Responda exclusivamente em JSON válido:

{
  "components": [
    {"name": "...", "type": "..."}
  ]
}
"""

response1 = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": free_extraction_prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

raw_components_text = response1.output_text
print(raw_components_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "User"},
    {"name": "AWS Shield", "type": "Security Service"},
    {"name": "Amazon CloudFront", "type": "Content Delivery Network"},
    {"name": "AWS WAF", "type": "Web Application Firewall"},
    {"name": "AWS Cloud", "type": "Cloud Infrastructure"},
    {"name": "sa-east-1 (São Paulo)", "type": "AWS Region"},
    {"name": "Virtual Private Cloud", "type": "Network"},
    {"name": "Availability Zone A", "type": "Availability Zone"},
    {"name": "Availability Zone B", "type": "Availability Zone"},
    {"name": "Availability Zone C", "type": "Availability Zone"},
    {"name": "Public Subnet", "type": "Subnet"},
    {"name": "Private Subnet", "type": "Subnet"},
    {"name": "Application Load Balancer", "type": "Load Balancer"},
    {"name": "SEI / SIP (Auto Scaling API Server)", "type": "Compute Instance"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "Storage"},
    {"name": "Amazon RDS (Primary)",

In [11]:
# Limpeza + Parsing

clean_text = raw_components_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

raw_components = json.loads(clean_text)["components"]

raw_components


[{'name': 'Usuários SEI', 'type': 'User'},
 {'name': 'AWS Shield', 'type': 'Security Service'},
 {'name': 'Amazon CloudFront', 'type': 'Content Delivery Network'},
 {'name': 'AWS WAF', 'type': 'Web Application Firewall'},
 {'name': 'AWS Cloud', 'type': 'Cloud Infrastructure'},
 {'name': 'sa-east-1 (São Paulo)', 'type': 'AWS Region'},
 {'name': 'Virtual Private Cloud', 'type': 'Network'},
 {'name': 'Availability Zone A', 'type': 'Availability Zone'},
 {'name': 'Availability Zone B', 'type': 'Availability Zone'},
 {'name': 'Availability Zone C', 'type': 'Availability Zone'},
 {'name': 'Public Subnet', 'type': 'Subnet'},
 {'name': 'Private Subnet', 'type': 'Subnet'},
 {'name': 'Application Load Balancer', 'type': 'Load Balancer'},
 {'name': 'SEI / SIP (Auto Scaling API Server)', 'type': 'Compute Instance'},
 {'name': 'Amazon Elastic File System (NFS) Multi-AZ', 'type': 'Storage'},
 {'name': 'Amazon RDS (Primary)', 'type': 'Database'},
 {'name': 'Amazon RDS (Secondary)', 'type': 'Database'

In [12]:
# Normalização Taxonômica

normalization_prompt = f"""
Normalize os componentes abaixo em categorias de segurança.

Categorias permitidas (escolha apenas UMA por item):

- user
- server
- database
- api
- external_system

Componentes:

{json.dumps(raw_components, indent=2)}

Responda em JSON válido:

{{
  "components": [
    {{"name": "...", "type": "..."}}
  ]
}}
"""

response2 = client.responses.create(
    model="gpt-4.1-mini",
    input=normalization_prompt
)

normalized_text = response2.output_text
print(normalized_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "user"},
    {"name": "AWS Shield", "type": "external_system"},
    {"name": "Amazon CloudFront", "type": "external_system"},
    {"name": "AWS WAF", "type": "external_system"},
    {"name": "AWS Cloud", "type": "external_system"},
    {"name": "sa-east-1 (São Paulo)", "type": "external_system"},
    {"name": "Virtual Private Cloud", "type": "external_system"},
    {"name": "Availability Zone A", "type": "external_system"},
    {"name": "Availability Zone B", "type": "external_system"},
    {"name": "Availability Zone C", "type": "external_system"},
    {"name": "Public Subnet", "type": "external_system"},
    {"name": "Private Subnet", "type": "external_system"},
    {"name": "Application Load Balancer", "type": "server"},
    {"name": "SEI / SIP (Auto Scaling API Server)", "type": "api"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "external_system"},
    {"name": "Amazon RDS (Primary)", "type": "d

In [13]:
# Parsing Normalizado

clean_text = normalized_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

components = json.loads(clean_text)["components"]

components


[{'name': 'Usuários SEI', 'type': 'user'},
 {'name': 'AWS Shield', 'type': 'external_system'},
 {'name': 'Amazon CloudFront', 'type': 'external_system'},
 {'name': 'AWS WAF', 'type': 'external_system'},
 {'name': 'AWS Cloud', 'type': 'external_system'},
 {'name': 'sa-east-1 (São Paulo)', 'type': 'external_system'},
 {'name': 'Virtual Private Cloud', 'type': 'external_system'},
 {'name': 'Availability Zone A', 'type': 'external_system'},
 {'name': 'Availability Zone B', 'type': 'external_system'},
 {'name': 'Availability Zone C', 'type': 'external_system'},
 {'name': 'Public Subnet', 'type': 'external_system'},
 {'name': 'Private Subnet', 'type': 'external_system'},
 {'name': 'Application Load Balancer', 'type': 'server'},
 {'name': 'SEI / SIP (Auto Scaling API Server)', 'type': 'api'},
 {'name': 'Amazon Elastic File System (NFS) Multi-AZ',
  'type': 'external_system'},
 {'name': 'Amazon RDS (Primary)', 'type': 'database'},
 {'name': 'Amazon RDS (Secondary)', 'type': 'database'},
 {'nam

In [14]:
stride_map = {
    "server": ["Spoofing", "Tampering", "Denial of Service"],
    "database": ["Tampering", "Information Disclosure"],
    "api": ["Spoofing", "Repudiation"],
    "user": ["Spoofing"],
    "external_system": ["Spoofing", "Tampering"]
}

def analyze_stride(components):
    results = []

    for comp in components:
        threats = stride_map.get(comp["type"])
        if threats is None:
            threats = ["Unknown – No STRIDE mapping defined"]

        results.append({
            "component": comp["name"],
            "type": comp["type"],
            "threats": threats
        })

    return results

stride_results = analyze_stride(components)

stride_results


[{'component': 'Usuários SEI', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'AWS Shield',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Amazon CloudFront',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'AWS WAF',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'AWS Cloud',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'sa-east-1 (São Paulo)',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Virtual Private Cloud',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Availability Zone A',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Availability Zone B',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Availability Zone C',
  'type': 'external_system',
  'threats': ['Spoofing',

In [15]:
enrichment_prompt = f"""
Considere os resultados de STRIDE abaixo:

{json.dumps(stride_results, indent=2)}

Para cada ameaça, explique o racional técnico e sugira contramedidas.

Resposta em texto estruturado.
"""

response3 = client.responses.create(
    model="gpt-4.1-mini",
    input=enrichment_prompt
)

print(response3.output_text)


Abaixo está a análise técnica para cada tipo de ameaça identificada no modelo STRIDE e respectivas sugestões de contramedidas.

---

## 1. Spoofing (Falsificação de Identidade)

### Racional Técnico
Spoofing ocorre quando um atacante consegue se passar por um usuário, sistema ou componente confiável, obtendo acesso não autorizado. No contexto dos componentes AWS e do sistema SEI, isso pode incluir:

- Falsificação de credenciais de usuário SEI.
- Ataques onde um invasor falsifica identidades em serviços AWS como CloudFront, WAF, VPC, Load Balancer, ou APIs.
- Mascaramento do endereço IP ou identidade da origem para enganar controles de segurança.

### Exemplos
- Um invasor que obtém credenciais válidas para se passar por um usuário SEI.
- Um serviço externo AWS que aceita solicitações de origens não autorizadas.
- APIs recebendo chamadas com tokens ou certificados falsificados.

### Sugestões de Contramedidas
- Implementar autenticação forte multifator (MFA) para usuários SEI.
- Utiliz

In [19]:
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm

def generate_pdf_report(stride_results, filename="relatorio_stride.pdf"):

    doc = SimpleDocTemplate(f"/content/{filename}", pagesize=A4)
    styles = getSampleStyleSheet()

    title = styles['Title']
    heading = styles['Heading2']
    body = styles['BodyText']

    elements = []

    elements.append(Paragraph("Relatório de Modelagem de Ameaças – STRIDE", title))
    elements.append(Spacer(1, 12))

    elements.append(Paragraph("1. Visão Geral", heading))
    elements.append(Paragraph(
        "Este relatório apresenta os resultados da análise automatizada de ameaças "
        "a partir de diagrama de arquitetura de software, utilizando LLM multimodal "
        "para extração de componentes e regras STRIDE para identificação de riscos.",
        body
    ))
    elements.append(Spacer(1, 6))

    elements.append(Paragraph("2. Componentes Identificados e Ameaças", heading))

    for item in stride_results:

        elements.append(Paragraph(f"<b>Componente:</b> {item['component']}", body))
        elements.append(Paragraph(f"<b>Tipo:</b> {item['type']}", body))
        elements.append(Paragraph(
            f"<b>Ameaças STRIDE:</b> {', '.join(item['threats'])}",
            body
        ))
        elements.append(Spacer(1, 6))

    elements.append(Paragraph("3. Conclusão", heading))
    elements.append(Paragraph(
        "A análise demonstra a viabilidade de utilização de modelos fundacionais "
        "multimodais como mecanismo de apoio à modelagem de ameaças em arquiteturas "
        "de software, reduzindo complexidade e custo de implementação de MVPs.",
        body
    ))

    doc.build(elements)

    return f"/content/{filename}"

In [20]:
pdf_path = generate_pdf_report(stride_results)

pdf_path


'/content/relatorio_stride.pdf'

In [21]:
from google.colab import files
files.download(pdf_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>